In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)
## knockout the btuB gene
model0['lb_list']['EX_cbl1_e ']=0
model0['lb_list']['EX_cbl1_e ']=0

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print(distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
    
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
        
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0]
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        save_rate = 0.9
                        while save_rate >= 0:
                            biomass_value2 = save_rate * biomass_value
                            try:
                                MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                            except:
                                save_rate = save_rate - 0.1
                            else:
                                #calculate the MDF values of metabolic networks
                                biomass_value_micro[i] = biomass_value2
                                B_value2 = MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                                #calculate the biomass yield under the MDF value
                                obj_name=biomass_list[i]
                                obj_target='maximize'
                                if i == 0:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                elif i == 1:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                biomass_value=max_biomass_under_mdf*0.9
                                
                                #calculate the minimum value of the sum of the fluxes
                                if i == 0:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                elif i == 1:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                
                                #translating the results of ETMs into a usable form
                                model=model_list[i]['model']
                                reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                                reaction_g0=model_list[i]['reaction_g0']
                                coef_matrix=model_list[i]['coef_matrix']
                                metabolite_list=model_list[i]['metabolite_list']
                                use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
                                distribute_public_metabolism_increase = {}
                                distribute_public_metabolism_decrease = {}
                                
                                #simulate the fluxes of the public metabolites
                                for rea in public_reaction_list[i]:
                                    for met in public_metabolism_name_list:
                                        try:
                                            model_list[i]['coef_matrix'][(met,rea)]
                                        except:
                                            pass
                                        else:
                                            public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                                            
                                #simulate the distribution of the public metabolites
                                for met in public_metabolism_name_list:
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]*save_rate
                                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
                                    
                                    
                                save_rate_ub = [save_rate]
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        met = rea[3::]
                                        if met in public_metabolism_name_list:
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            save_rate_met = distribute_public_metabolism_list[met][0,m]/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                            save_rate_ub.append(save_rate_met)
                                        
                                save_rate_final = min(save_rate_ub)
                                
                                
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        if met in public_metabolism_name_list:
                                            met = rea[3::]
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - intracellular_c_dict[met]*int((save_rate_final)*distribute_micro_list_t[ct][i][m]) + intracellular_c_dict[met]*int((1-save_rate_final)*distribute_micro_list_t[ct][i][m])
                                            if distribute_public_metabolism_list[met][0,m] < 0:
                                                print(met + '_3: ', distribute_public_metabolism_list[met][0,m])
                                    
                                
                                distribute_micro_decrease = int((1-save_rate_final) * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_increase = int(save_rate_final * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] - distribute_micro_decrease + distribute_micro_increase)
                                
                                break
                            
                        continue
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
            
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
            
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
            
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])
                                        
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death        
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                        
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                                
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
                
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                                    if i == 0:
                                        # gene upregulation for Fe3+ intake
                                        if 'EX_fe3_e' in rea:
                                            public_reaction_ub_list[i][rea][0, m] = public_reaction_ub_list[i][rea][0, m] * 1.2
        
        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
                                    
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
        
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([30, 33, 21,  3, 16, 30, 30, 16, 20,  8, 23, 39, 35, 14, 35,  4, 27,
       14, 21]), 1: array([ 7, 20,  1, 17, 28,  3, 21, 19,  3, 21, 15,  0, 25, 37,  3, 18, 38,
       35, 10])}
1
strain_number:  0 26.105263157894736
strain_various:  0 12.615340439497944
strain_number:  1 17.05263157894737
strain_various:  1 12.14592712419975
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.94736842105263
strain_various:  0 8.500611025781284
strain_number:  1 17.05263157894737
strain_various:  1 6.4438766054581755
fd_r_threshold:  [1.05, 204.96306717177868]
slip_r:  41.832613434355736
3
strain_number:  0 36.78947368421053
strain_various:  0 8.185860385402837
strain_number:  1 17.05263157894737
strain_various:  1 5.799035166669187
fd_r_threshold:  [1.05, 204.96306717177868, 10.709251086952904]
slip_r:  43.76446365174631
4
strain_number:  0 43.73684210526316
strain_various:  0 9.255678798861856
strain_number:  1 17.105263157894736
strain_various:  1 5.92847488013841
fd_r_threshol

glc__D_e_1:  -4.045863536005739
glc__D_e_1:  -3.149748813579804
glc__D_e_1:  -7.139680716675478
glc__D_e_1:  -11.23959466246899
glc__D_e_1:  -6.660669045900085
glc__D_e_1:  -3.2934424914580798
glc__D_e_1:  -4.388211036328013
glc__D_e_1:  -7.6419312415766845
glc__D_e_1:  -6.04659066233941
glc__D_e_1:  -2.8483285549785653
glc__D_e_1:  -5.362882902076793
glc__D_e_1:  -12.63853323960319
glc__D_e_1:  -11.917207491828554
glc__D_e_1:  -5.687616308794473
glc__D_e_1:  -6.782384853664406
glc__D_e_1:  -7.032717709788567
glc__D_e_1:  -6.929169649217108
strain_number:  0 14.157894736842104
strain_various:  0 4.880284265822581
strain_number:  1 6.7368421052631575
strain_various:  1 2.917138202221267
fd_r_threshold:  [1.05, 204.96306717177868, 10.709251086952904, 8.06023974093556, 7.076186845878628, 6.57894214353639, 5.238623512115361, 3.789245501595197, 4.055940990191857, 4.498906507269213, 4.262736668784327, 4.317138646522896]
slip_r:  4.184793662872698
13
glc__D_e_1:  -10.742575461329807
glc__D_e_

glc__D_e_1:  -1.3163203971809958
glc__D_e_1:  -0.5567712084114333
glc__D_e_1:  -1.2470584794353934
glc__D_e_1:  -0.17543129189110473
glc__D_e_1:  -0.3378934974154695
glc__D_e_1:  -0.6893975398523674
glc__D_e_1:  -0.49564330023026115
glc__D_e_1:  -1.173964567210257
glc__D_e_1:  -1.0195201454094631
glc__D_e_1:  -0.871548625257242
glc__D_e_1:  -0.43522090845352457
glc__D_e_1:  -0.3356371688296247
glc__D_e_1:  -0.1811927470288306
glc__D_e_1:  -0.04511266528420399
glc__D_e_1:  -0.5022361142564782
glc__D_e_1:  -0.1138072339292393
glc__D_e_1:  -1.2337178185487634
glc__D_e_1:  -1.74794744518048
strain_number:  0 2.3684210526315788
strain_various:  0 0.871207650381413
strain_number:  1 1.1578947368421053
strain_various:  1 0.6698906348083082
fd_r_threshold:  [1.05, 204.96306717177868, 10.709251086952904, 8.06023974093556, 7.076186845878628, 6.57894214353639, 5.238623512115361, 3.789245501595197, 4.055940990191857, 4.498906507269213, 4.262736668784327, 4.317138646522896, 4.512637562285254, 3.880

strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 204.96306717177868, 10.709251086952904, 8.06023974093556, 7.076186845878628, 6.57894214353639, 5.238623512115361, 3.789245501595197, 4.055940990191857, 4.498906507269213, 4.262736668784327, 4.317138646522896, 4.512637562285254, 3.880368375018934, 4.507541729654824, 4.237692743764173, 5.234965986394556, 4.080277777777778, 8.284722222222221, 4.083333333333334, 5.25, 4.0, 5.472222222222222, 2.5, 6.0, 1.0, 1.25, 0.0, 0.0]
slip_r:  1.65
30
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 204.96306717177868, 10.709251086952904, 8.06023974093556, 7.076186845878628, 6.57894214353639, 5.238623512115361, 3.789245501595197, 4.055940990191857, 4.498906507269213, 4.262736668784327, 4.317138646522896, 4.512637562285254, 3.8803683

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([11, 37, 10,  0, 12, 13, 35,  5, 31, 34,  6, 35, 20, 22, 22, 17, 27,
       21, 12]), 1: array([ 1, 35, 11,  8, 24,  7, 34, 27,  9, 28, 15, 16, 32, 26, 12,  0, 11,
       27,  6])}
1
strain_number:  0 23.105263157894736
strain_various:  0 13.142517330133263
strain_number:  1 17.473684210526315
strain_various:  1 11.240754896230017
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 27.36842105263158
strain_various:  0 9.257773554755794
strain_number:  1 17.473684210526315
strain_various:  1 5.669571793471676
fd_r_threshold:  [1.05, 229.22656478945353]
slip_r:  46.6853129578907
3
strain_number:  0 32.526315789473685
strain_various:  0 8.475155151833235
strain_number:  1 17.473684210526315
strain_various:  1 4.2347984610297775
fd_r_threshold:  [1.05, 229.22656478945353, 10.053665230023496]
slip_r:  48.4860460038954
4
strain_number:  0 38.68421052631579
strain_various:  0 10.208627061558985
strain_number:  1 17.473684210526315
strain_various:  1 3.8439072918968873
fd_r_thr

glc__D_e_1:  -3.111537176274969
glc__D_e_1:  -2.7813066244244258
glc__D_e_1:  -10.213786158180808
glc__D_e_1:  -18.857163836965988
glc__D_e_1:  -6.819275627068006
strain_number:  0 16.789473684210527
strain_various:  0 6.15226913524314
strain_number:  1 7.105263157894737
strain_various:  1 2.268659164447611
fd_r_threshold:  [1.05, 229.22656478945353, 10.053665230023496, 8.008236610590508, 7.695818215705298, 6.896160839841408, 6.008718094071897, 3.576386327732517, 4.01040683116279, 5.046596696385268, 4.6104043075724865, 4.256340423814393]
slip_r:  4.300026917333491
13
glc__D_e_1:  -11.328439913453598
glc__D_e_1:  -4.074818103154651
glc__D_e_1:  -1.181839213180094
glc__D_e_1:  -3.6963935602783216
glc__D_e_1:  -11.356964666831814
glc__D_e_1:  -6.109365001830398
glc__D_e_1:  -2.6207354176236
glc__D_e_1:  -5.627082283387643
glc__D_e_1:  -7.0242624874505495
glc__D_e_1:  -5.583366330014069
glc__D_e_1:  -6.128260575350536
glc__D_e_1:  -10.173784232215361
glc__D_e_1:  -4.760913094018309
glc__D_

glc__D_e_1:  -0.24916106925761028
glc__D_e_1:  -0.2942201884603435
glc__D_e_1:  -0.13977576665954938
glc__D_e_1:  -0.01860085820189572
glc__D_e_1:  -0.3335267123727548
glc__D_e_1:  -0.1790822905719609
glc__D_e_1:  -0.8407391960482888
glc__D_e_1:  -1.1897892594857085
glc__D_e_1:  -1.7040188861174252
glc__D_e_1:  -0.6185681768704656
glc__D_e_1:  -0.9609536697581451
glc__D_e_1:  -0.13783519952484047
glc__D_e_1:  -0.40937635467844147
glc__D_e_1:  -0.6025687751125277
glc__D_e_1:  -0.5269087709080009
glc__D_e_1:  -0.5040350205226438
glc__D_e_1:  -1.4063251033670676
glc__D_e_1:  -0.2296372811270493
glc__D_e_1:  -0.5898590395356322
glc__D_e_1:  -0.43541461773483825
strain_number:  0 2.1578947368421053
strain_various:  0 0.9874559494365115
strain_number:  1 1.4736842105263157
strain_various:  1 0.7517293082676685
fd_r_threshold:  [1.05, 229.22656478945353, 10.053665230023496, 8.008236610590508, 7.695818215705298, 6.896160839841408, 6.008718094071897, 3.576386327732517, 4.01040683116279, 5.04659

glc__D_e_1:  -0.06565357136788263
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 229.22656478945353, 10.053665230023496, 8.008236610590508, 7.695818215705298, 6.896160839841408, 6.008718094071897, 3.576386327732517, 4.01040683116279, 5.046596696385268, 4.6104043075724865, 4.256340423814393, 4.181643173336571, 4.05210474930713, 3.5322180227010813, 4.407251984126985, 4.860277777777778, 4.354444444444445, 3.845555555555556, 6.361111111111111, 8.0, 6.0, 5.0, 1.0, 1.0, 1.0, 4.0, 1.0, 2.0]
slip_r:  1.8
30
glc__D_e_1:  -0.029452032311810095
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 229.22656478945353, 10.053665230023496, 8.008236610590508, 7.695818215705298, 6.896160839841408, 6.008718094071897, 3.576386327732517, 4.01040683116279, 5.046596696385268, 4.6104043075724865, 4.256340423814393, 4.181

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([26, 33, 25, 28, 36, 14, 12, 29, 39, 19, 26, 19, 14, 31, 19, 11, 16,
       38,  2]), 1: array([32, 30, 23, 12,  7,  2, 24, 17, 12, 31, 32, 34, 25, 39, 34, 28, 14,
       32, 10])}
1
strain_number:  0 27.105263157894736
strain_various:  0 11.880522480279733
strain_number:  1 23.473684210526315
strain_various:  1 10.93698900666248
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.21052631578947
strain_various:  0 8.198709445095426
strain_number:  1 23.68421052631579
strain_various:  1 8.897383598735477
fd_r_threshold:  [1.05, 94.21056783215457]
slip_r:  19.682113566430914
3
strain_number:  0 38.26315789473684
strain_various:  0 9.737837786929033
strain_number:  1 24.0
strain_various:  1 7.9339377626152725
fd_r_threshold:  [1.05, 94.21056783215457, 8.901790550366096]
slip_r:  21.252471676504136
4
strain_number:  0 45.526315789473685
strain_various:  0 11.550203557343329
strain_number:  1 24.210526315789473
strain_various:  1 7.323925804967982
fd_r_threshold:  [1.05, 

glc__D_e_1:  -4.042805386105583
glc__D_e_1:  -4.248146169135712
glc__D_e_1:  -2.7543355376289043
glc__D_e_1:  -5.760682403392947
glc__D_e_1:  -7.710100363801452
glc__D_e_1:  -4.1087376392666455
glc__D_e_1:  -0.5581411379282599
glc__D_e_1:  -2.0891104476948574
glc__D_e_1:  -13.11419194083935
glc__D_e_1:  -9.04949595090216
glc__D_e_1:  -0.33014051204225625
glc__D_e_1:  -0.3857322658114075
glc__D_e_1:  -8.169865520139926
glc__D_e_1:  -5.9058508924701405
glc__D_e_1:  -4.591941084450459
glc__D_e_1:  -6.178502147986207
glc__D_e_1:  -8.370851398401937
glc__D_e_1:  -7.444184867597174
glc__D_e_1:  -2.0590144770775867
glc__D_e_1:  -1.6228137121809225
glc__D_e_1:  -7.116447669631307
glc__D_e_1:  -7.681573657492358
glc__D_e_1:  -2.381522277544171
glc__D_e_1:  -2.4371140313133224
glc__D_e_1:  -7.379417340960856
glc__D_e_1:  -7.790098907021114
glc__D_e_1:  -5.163850213200665
glc__D_e_1:  -4.291448683407337
glc__D_e_1:  -6.647941414806621
glc__D_e_1:  -7.881741451100183
glc__D_e_1:  -8.62867303730116

glc__D_e_1:  -0.3760975583175086
glc__D_e_1:  -0.4455378859770778
glc__D_e_1:  -0.2910934641762837
glc__D_e_1:  -0.1329793637831218
glc__D_e_1:  -0.4014172048732241
glc__D_e_1:  -0.9156468315049406
glc__D_e_1:  -0.8000026424849094
glc__D_e_1:  -0.6535708612153626
glc__D_e_1:  -1.83647453627959
glc__D_e_1:  -1.343717570339118
glc__D_e_1:  -1.767881954042656
glc__D_e_1:  -0.36894155632949
glc__D_e_1:  -2.321759384517816
strain_number:  0 2.736842105263158
strain_various:  0 0.96475277788544
strain_number:  1 1.736842105263158
strain_various:  1 1.017846295042827
fd_r_threshold:  [1.05, 94.21056783215457, 8.901790550366096, 8.420273335743818, 8.025037242932433, 6.862629353219199, 5.2204984062886135, 3.7987989163959903, 5.376241141900297, 4.448741913788377, 5.946341385801612, 5.192230708140505, 4.40557206180895, 3.344762189670199, 4.06330010707987, 4.874506802721088, 4.241944444444445]
slip_r:  4.186017121144911
18
glc__D_e_1:  -0.5135534650173097
glc__D_e_1:  -0.6871335073653626
glc__D_e_

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 94.21056783215457, 8.901790550366096, 8.420273335743818, 8.025037242932433, 6.862629353219199, 5.2204984062886135, 3.7987989163959903, 5.376241141900297, 4.448741913788377, 5.946341385801612, 5.192230708140505, 4.40557206180895, 3.344762189670199, 4.06330010707987, 4.874506802721088, 4.241944444444445, 4.053888888888889, 4.611111111111111, 6.361111111111111, 6.111111111111111, 2.5, 4.0, 4.0, 9.0, 0.0, 0.0]
slip_r:  3.4
28
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 94.21056783215457, 8.901790550366096, 8.420273335743818, 8.025037242932433, 6.862629353219199, 5.2204984062886135, 3.7987989163959903, 5.376241141900297, 4.448741913788377, 5.946341385801612, 5.192230708140505, 4.4055720

strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 94.21056783215457, 8.901790550366096, 8.420273335743818, 8.025037242932433, 6.862629353219199, 5.2204984062886135, 3.7987989163959903, 5.376241141900297, 4.448741913788377, 5.946341385801612, 5.192230708140505, 4.40557206180895, 3.344762189670199, 4.06330010707987, 4.874506802721088, 4.241944444444445, 4.053888888888889, 4.611111111111111, 6.361111111111111, 6.111111111111111, 2.5, 4.0, 4.0, 9.0, 0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
slip_r:  0.0
The change process of microorganism distribution is:  {0: {0: array([26, 33, 25, 28, 36, 14, 12, 29, 39, 19, 26, 19, 14, 31, 19, 11, 16,
       38,  2]), 1: array([32, 30, 23, 12,  7,  2, 24, 17, 12, 31, 32, 34, 25, 39, 34, 28, 14,
       32, 10])}, 1: {0: array([31, 39, 30, 33, 43, 16, 14, 34, 46, 22, 31, 22, 16, 37, 22, 13, 19,
       45,  2]), 1: array([33

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([20,  2, 39,  9, 24, 34, 34, 35, 19, 38,  9, 16, 21, 16, 20, 13, 21,
       38, 31]), 1: array([ 5, 35,  0, 38, 30, 11, 29, 18, 35, 30,  3,  7, 19,  9,  3, 13, 14,
       11, 16])}
1
strain_number:  0 27.263157894736842
strain_various:  0 12.97301695732605
strain_number:  1 17.42105263157895
strain_various:  1 12.205761320715826
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.36842105263158
strain_various:  0 10.820110546744305
strain_number:  1 17.473684210526315
strain_various:  1 7.6939376069914305
fd_r_threshold:  [1.05, 475.9215973356989]
slip_r:  96.02431946713978
3
strain_number:  0 38.473684210526315
strain_various:  0 9.427666011616703
strain_number:  1 17.57894736842105
strain_various:  1 6.900158575398899
fd_r_threshold:  [1.05, 475.9215973356989, 21.09347367552265]
slip_r:  100.0330142022443
4
strain_number:  0 45.78947368421053
strain_various:  0 9.550855279245779
strain_number:  1 17.63157894736842
strain_various:  1 6.276191657163723
fd_r_threshold

glc__D_e_1:  -8.44523967013971
glc__D_e_1:  -3.50361059952352
glc__D_e_1:  -5.090171663059268
glc__D_e_1:  -5.496821250330676
glc__D_e_1:  -4.210369514694989
glc__D_e_1:  -3.1001348936665023
glc__D_e_1:  -4.139311684767284
glc__D_e_1:  -8.36107820525898
glc__D_e_1:  -8.103085722886728
glc__D_e_1:  -1.401892525172447
glc__D_e_1:  -1.4574842789415983
glc__D_e_1:  -8.610752737214884
glc__D_e_1:  -6.346738109545099
glc__D_e_1:  -4.616629701584943
glc__D_e_1:  -8.170360839783953
glc__D_e_1:  -10.487574738451283
glc__D_e_1:  -3.9026269765848456
glc__D_e_1:  -3.367624209287442
glc__D_e_1:  -6.921355347486452
glc__D_e_1:  -11.946134186970559
glc__D_e_1:  -4.538067954870817
glc__D_e_1:  -3.0192201360861963
glc__D_e_1:  -9.959907151176761
glc__D_e_1:  -10.866820096328116
glc__D_e_1:  -3.613198286029168
glc__D_e_1:  -2.4633174874057104
glc__D_e_1:  -8.912211983830462
glc__D_e_1:  -15.822663985975039
glc__D_e_1:  -4.093664619678645
glc__D_e_1:  -1.9177985642546478
glc__D_e_1:  -7.383108023347768
g

glc__D_e_1:  -0.4286265118242847
glc__D_e_1:  -1.054240983487759
glc__D_e_1:  -0.23112251325445443
strain_number:  0 3.3684210526315788
strain_various:  0 0.9846466807299845
strain_number:  1 1.105263157894737
strain_various:  1 0.5520046569316588
fd_r_threshold:  [1.05, 475.9215973356989, 21.09347367552265, 7.034180626492994, 7.7402253830107295, 9.383317260414056, 7.114144478994016, 4.825252078773827, 4.984587113750253, 4.665818225090254, 4.051676953344998, 4.137107314417013, 3.9462736216774075, 4.0162199625220465, 2.943532936627759, 5.666289997480474, 4.851354875283447]
slip_r:  4.284734278718227
18
glc__D_e_1:  -0.7931056267445208
glc__D_e_1:  -0.6386612049437268
glc__D_e_1:  -0.8242512251054457
glc__D_e_1:  -0.135193778909523
glc__D_e_1:  -1.1784820427776008
glc__D_e_1:  -0.41328907007458726
glc__D_e_1:  -0.4375561943715991
glc__D_e_1:  -0.31715387656758187
glc__D_e_1:  -0.7018777088710884
glc__D_e_1:  -1.1121220499506754
glc__D_e_1:  -0.2890035797173708
glc__D_e_1:  -0.70178268328

strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 475.9215973356989, 21.09347367552265, 7.034180626492994, 7.7402253830107295, 9.383317260414056, 7.114144478994016, 4.825252078773827, 4.984587113750253, 4.665818225090254, 4.051676953344998, 4.137107314417013, 3.9462736216774075, 4.0162199625220465, 2.943532936627759, 5.666289997480474, 4.851354875283447, 5.323333333333333, 4.069444444444445, 7.833333333333334, 4.75, 6.5, 2.0, 2.0, 0.0, 3.0, 5.0, 1.0]
slip_r:  2.2
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 475.9215973356989, 21.09347367552265, 7.034180626492994, 7.7402253830107295, 9.383317260414056, 7.114144478994016, 4.825252078773827, 4.984587113750253, 4.665818225090254, 4.051676953344998, 4.137107314417013, 3.9462736216774075, 4.0162199625220465, 2.943532936627759, 5.666289997480474,

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([23, 23, 22,  6, 19,  3,  4, 20, 22, 25, 28, 38, 29, 38, 13, 19,  3,
       32, 16]), 1: array([36,  6, 34, 11, 23, 31,  9, 12, 11, 16, 11, 30, 27,  0,  5,  3, 31,
       12, 21])}
1
strain_number:  0 23.68421052631579
strain_various:  0 12.574211837093399
strain_number:  1 17.57894736842105
strain_various:  1 11.444676029415712
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.0
strain_various:  0 11.589468903866312
strain_number:  1 17.57894736842105
strain_various:  1 6.12428977170057
fd_r_threshold:  [1.05, 129.11452435979163]
slip_r:  26.662904871958325
3
strain_number:  0 33.21052631578947
strain_various:  0 12.193954271507799
strain_number:  1 17.63157894736842
strain_various:  1 5.101192082721444
fd_r_threshold:  [1.05, 129.11452435979163, 8.624858333147916]
slip_r:  28.177876538587913
4
strain_number:  0 39.36842105263158
strain_various:  0 14.246737489189327
strain_number:  1 17.68421052631579
strain_various:  1 4.984741260021716
fd_r_threshold:  [1.05, 1

glc__D_e_1:  -1.643646226961701
glc__D_e_1:  -2.1910304993966676
glc__D_e_1:  -15.29520756751886
glc__D_e_1:  -6.2409043949525085
glc__D_e_1:  -2.0130917652012954
glc__D_e_1:  -5.511231149631154
glc__D_e_1:  -15.253124492422215
glc__D_e_1:  -6.867495368288374
glc__D_e_1:  -1.9331488962271894
glc__D_e_1:  -4.447703243325417
glc__D_e_1:  -18.884430549276566
glc__D_e_1:  -8.852564484676115
strain_number:  0 15.473684210526315
strain_various:  0 6.9081829513809705
strain_number:  1 7.157894736842105
strain_various:  1 3.406853122669949
fd_r_threshold:  [1.05, 129.11452435979163, 8.624858333147916, 9.326103722582596, 16.821216806144587, 16.16366888077397, 9.147436228404286, 6.281659260790246, 5.699403221775457, 4.518940577191361, 4.750755941103044, 4.163235740482683]
slip_r:  5.082798948268558
13
glc__D_e_1:  -4.112637533001209
glc__D_e_1:  -4.317978316031338
glc__D_e_1:  -4.473019831579421
glc__D_e_1:  -4.092410820451908
glc__D_e_1:  -5.450646147073343
glc__D_e_1:  -5.501542508302677
glc__

glc__D_e_1:  -0.575076619509679
glc__D_e_1:  -0.12709204090110093
glc__D_e_1:  -0.6413216675328175
glc__D_e_1:  -0.6380691811498476
glc__D_e_1:  -1.1646473297318538
glc__D_e_1:  -2.347551004796081
glc__D_e_1:  -0.697751758057173
glc__D_e_1:  -1.2119813846888898
glc__D_e_1:  -1.2337096727001584
glc__D_e_1:  -1.5838444822304232
glc__D_e_1:  -0.8750655140648884
glc__D_e_1:  -0.9243523503632985
glc__D_e_1:  -0.1012338801299939
glc__D_e_1:  -0.5797644524895982
glc__D_e_1:  -0.9668546158796665
glc__D_e_1:  -0.14373614564636183
glc__D_e_1:  -0.017052642012691965
glc__D_e_1:  -0.7484454531931646
strain_number:  0 2.8947368421052633
strain_various:  0 1.1189627171299632
strain_number:  1 0.8947368421052632
strain_various:  1 0.7877173445839877
fd_r_threshold:  [1.05, 129.11452435979163, 8.624858333147916, 9.326103722582596, 16.821216806144587, 16.16366888077397, 9.147436228404286, 6.281659260790246, 5.699403221775457, 4.518940577191361, 4.750755941103044, 4.163235740482683, 4.783877803861624, 4

strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 129.11452435979163, 8.624858333147916, 9.326103722582596, 16.821216806144587, 16.16366888077397, 9.147436228404286, 6.281659260790246, 5.699403221775457, 4.518940577191361, 4.750755941103044, 4.163235740482683, 4.783877803861624, 4.301942838593324, 5.767117112985619, 5.290610860199605, 4.590556972789115, 6.227222222222222, 5.206666666666667, 4.109999999999999, 6.222222222222222, 5.111111111111111, 3.25, 4.25, 4.25, 1.0, 1.0, 3.0, 0.0]
slip_r:  1.85
30
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 129.11452435979163, 8.624858333147916, 9.326103722582596, 16.821216806144587, 16.16366888077397, 9.147436228404286, 6.281659260790246, 5.699403221775457, 4.518940577191361, 4.750755941103044, 4.163235740482683, 4.783877803861624, 4.301942838593324, 5.76

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([34, 24,  3, 38, 23,  7, 39,  9, 38, 30,  5, 29, 23, 18, 26, 25,  6,
       26, 36]), 1: array([37, 30, 23, 19, 22, 12, 25, 27, 27, 28,  9, 12, 25, 28, 20, 29,  4,
       28, 27])}
1
strain_number:  0 27.263157894736842
strain_various:  0 13.969297557079495
strain_number:  1 22.842105263157894
strain_various:  1 8.209514107647085
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.36842105263158
strain_various:  0 5.630841073155566
strain_number:  1 22.94736842105263
strain_various:  1 5.144989225199974
fd_r_threshold:  [1.05, 124.29209424403366]
slip_r:  25.69841884880673
3
strain_number:  0 38.473684210526315
strain_various:  0 6.310963068157302
strain_number:  1 23.05263157894737
strain_various:  1 5.010238824127273
fd_r_threshold:  [1.05, 124.29209424403366, 5.632640054230996]
slip_r:  26.614946859652935
4
strain_number:  0 45.78947368421053
strain_various:  0 12.159375621818143
strain_number:  1 23.210526315789473
strain_various:  1 6.177880298787731
fd_r_thresh

glc__D_e_1:  -4.556160626719443
glc__D_e_1:  -4.607056987948777
glc__D_e_1:  -2.1875297152791076
glc__D_e_1:  -2.7349139877140742
glc__D_e_1:  -5.5475333410463055
glc__D_e_1:  -6.26710375070815
glc__D_e_1:  -4.140831635142003
glc__D_e_1:  -2.284845068017045
glc__D_e_1:  -11.818617374960237
glc__D_e_1:  -11.251736048986395
glc__D_e_1:  -2.244649375140242
glc__D_e_1:  -2.7920336475752086
glc__D_e_1:  -7.089486535543902
glc__D_e_1:  -6.317264426539932
glc__D_e_1:  -6.603551915665886
glc__D_e_1:  -10.157283053864894
glc__D_e_1:  -7.829465592907692
glc__D_e_1:  -5.565450965237907
glc__D_e_1:  -1.3390688846027008
glc__D_e_1:  -1.3946606383718523
glc__D_e_1:  -8.327992125629605
glc__D_e_1:  -6.0639774979598196
glc__D_e_1:  -7.042987531542909
glc__D_e_1:  -9.668725386179439
glc__D_e_1:  -6.358406258795538
glc__D_e_1:  -2.9114879560615257
glc__D_e_1:  -5.952303668035766
glc__D_e_1:  -7.047072212905699
glc__D_e_1:  -9.010079553061665
glc__D_e_1:  -7.260294552023597
glc__D_e_1:  -3.77876203131523

glc__D_e_1:  -1.0139461239671936
glc__D_e_1:  -0.8332427192246071
glc__D_e_1:  -0.4245963349150832
glc__D_e_1:  -0.2701519131142893
glc__D_e_1:  -0.8669856324962202
glc__D_e_1:  -2.7853373996363873
glc__D_e_1:  -2.476448556034799
glc__D_e_1:  -1.3708908090098006
glc__D_e_1:  -0.5246350603715455
glc__D_e_1:  -1.2082723177576877
glc__D_e_1:  -0.43967120148565697
glc__D_e_1:  -1.3130926469560609
glc__D_e_1:  -0.5053709752840609
glc__D_e_1:  -0.5604350213495022
strain_number:  0 3.0
strain_various:  0 1.1697953037312037
strain_number:  1 1.4736842105263157
strain_various:  1 0.6781104593013224
fd_r_threshold:  [1.05, 124.29209424403366, 5.632640054230996, 9.290275364095367, 11.233265305100703, 6.918437621144583, 4.216697551943684, 4.5573173852059705, 4.919369785037964, 4.570044826208809, 5.596829288578706, 3.9258802815044866, 6.3440924687068145, 4.065730753734001, 4.066240236835475, 3.156736111111112, 3.784297052154195]
slip_r:  4.283419324508319
18
glc__D_e_1:  -0.9412586680338051
glc__D_

strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 124.29209424403366, 5.632640054230996, 9.290275364095367, 11.233265305100703, 6.918437621144583, 4.216697551943684, 4.5573173852059705, 4.919369785037964, 4.570044826208809, 5.596829288578706, 3.9258802815044866, 6.3440924687068145, 4.065730753734001, 4.066240236835475, 3.156736111111112, 3.784297052154195, 3.6388888888888893, 7.006944444444445, 7.0, 6.472222222222222, 4.25, 1.0, 4.25, 4.0, 2.0, 3.0]
slip_r:  2.85
28
glc__D_e_1:  -0.09113270825892184
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 124.29209424403366, 5.632640054230996, 9.290275364095367, 11.233265305100703, 6.918437621144583, 4.216697551943684, 4.5573173852059705, 4.919369785037964, 4.570044826208809, 5.596829288578706, 3.9258802815044866, 6.3440924687068145, 4.065730753734001, 4.

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([28,  1, 14, 14, 33, 28, 28, 29, 17, 33, 22, 36, 21,  9, 33,  3,  9,
       19, 12]), 1: array([ 9, 30, 14,  3,  3, 34, 21,  3, 19, 26, 24, 11,  6, 11, 12, 12, 35,
       10, 32])}
1
strain_number:  0 24.0
strain_various:  0 12.553045342402816
strain_number:  1 16.789473684210527
strain_various:  1 10.855893123666307
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.42105263157895
strain_various:  0 9.77814851419485
strain_number:  1 16.789473684210527
strain_various:  1 5.699297716371184
fd_r_threshold:  [1.05, 180.80721318211815]
slip_r:  37.001442636423626
3
strain_number:  0 33.73684210526316
strain_various:  0 11.077915941486836
strain_number:  1 16.789473684210527
strain_various:  1 4.237414149025335
fd_r_threshold:  [1.05, 180.80721318211815, 13.307931388478007]
slip_r:  39.45302891411923
4
strain_number:  0 40.10526315789474
strain_various:  0 11.849471393238138
strain_number:  1 16.789473684210527
strain_various:  1 3.874056046226936
fd_r_threshold:  [1.05

glc__D_e_1:  -7.297239508294648
glc__D_e_1:  -11.866120706881055
glc__D_e_1:  -4.458054474781314
glc__D_e_1:  -4.7190279069031345
glc__D_e_1:  -6.305588970438882
glc__D_e_1:  -11.782590746511588
glc__D_e_1:  -8.541013226807703
glc__D_e_1:  -3.5229878381025603
glc__D_e_1:  -4.125963864306678
glc__D_e_1:  -13.586801439345775
glc__D_e_1:  -10.705009124472813
strain_number:  0 15.842105263157896
strain_various:  0 5.696380733641659
strain_number:  1 6.684210526315789
strain_various:  1 2.9028594175026687
fd_r_threshold:  [1.05, 180.80721318211815, 13.307931388478007, 8.291098987719081, 8.200363817935862, 7.362256186333021, 6.5469853219340575, 4.033471737104616, 3.563281931111882, 4.5681584446319885, 4.634399128714872, 4.8781574369894445]
slip_r:  4.3354937357105605
13
glc__D_e_1:  -8.020333300030329
glc__D_e_1:  -3.081622478630501
glc__D_e_1:  -0.02296133942681733
glc__D_e_1:  -0.570345611861784
glc__D_e_1:  -9.419832690761464
glc__D_e_1:  -4.326677447560842
glc__D_e_1:  -3.387398918089227

glc__D_e_1:  -0.07975353107043848
glc__D_e_1:  -1.2051931354290404
glc__D_e_1:  -1.008602663960393
glc__D_e_1:  -0.8541582421595991
glc__D_e_1:  -0.2453271359390099
glc__D_e_1:  -0.20990820979025404
glc__D_e_1:  -0.17950232118491938
glc__D_e_1:  -0.6884949681524506
glc__D_e_1:  -0.42887169914171497
glc__D_e_1:  -0.2744272773409211
glc__D_e_1:  -0.9537418311272636
glc__D_e_1:  -0.13062336089395898
glc__D_e_1:  -0.008660110772223506
glc__D_e_1:  -0.3308474177942746
glc__D_e_1:  -1.0310243014364315
glc__D_e_1:  -0.8765798796356375
glc__D_e_1:  -0.6374991891198329
glc__D_e_1:  -1.2762877872546357
glc__D_e_1:  -1.2989109123122655
glc__D_e_1:  -1.813140538943982
strain_number:  0 2.6315789473684212
strain_various:  0 0.7405919620773835
strain_number:  1 1.0526315789473684
strain_various:  1 0.6862318321265946
fd_r_threshold:  [1.05, 180.80721318211815, 13.307931388478007, 8.291098987719081, 8.200363817935862, 7.362256186333021, 6.5469853219340575, 4.033471737104616, 3.563281931111882, 4.5681

strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 180.80721318211815, 13.307931388478007, 8.291098987719081, 8.200363817935862, 7.362256186333021, 6.5469853219340575, 4.033471737104616, 3.563281931111882, 4.5681584446319885, 4.634399128714872, 4.8781574369894445, 4.387161686378043, 3.7053003158592177, 4.9113328507819425, 4.475174495418003, 5.400028344671201, 3.929166666666667, 4.493055555555555, 7.222222222222222, 5.722222222222222, 2.611111111111111, 6.5, 3.25, 4.0, 0.0, 3.0, 1.0]
slip_r:  2.25
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 180.80721318211815, 13.307931388478007, 8.291098987719081, 8.200363817935862, 7.362256186333021, 6.5469853219340575, 4.033471737104616, 3.563281931111882, 4.5681584446319885, 4.634399128714872, 4.8781574369894445, 4.387161686378043, 3.7053003158592177, 4

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([17, 28, 36, 10, 15, 38, 24, 26, 18, 26, 33, 30, 14, 27, 13,  5, 18,
       26, 38]), 1: array([29, 37,  2, 26, 12,  4,  5, 17, 11, 17, 20, 11, 25, 20, 28, 35, 21,
       23, 13])}
1
strain_number:  0 27.526315789473685
strain_various:  0 11.160873485212015
strain_number:  1 18.842105263157894
strain_various:  1 9.91682864149165
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.578947368421055
strain_various:  0 7.169495243729609
strain_number:  1 18.894736842105264
strain_various:  1 7.047916727225558
fd_r_threshold:  [1.05, 49.49285133007527]
slip_r:  10.738570266015055
3
strain_number:  0 38.68421052631579
strain_various:  0 7.881040751328451
strain_number:  1 18.894736842105264
strain_various:  1 5.683723175988979
fd_r_threshold:  [1.05, 49.49285133007527, 7.4326792874974705]
slip_r:  12.015106123514547
4
strain_number:  0 45.94736842105263
strain_various:  0 8.666169458001168
strain_number:  1 19.0
strain_various:  1 6.087087282021457
fd_r_threshold:  [1.05, 4

glc__D_e_1:  -4.772696837269095
glc__D_e_1:  -14.921526760028307
glc__D_e_1:  -9.210593829624509
glc__D_e_1:  -0.6559958549862315
glc__D_e_1:  -2.186965164752829
glc__D_e_1:  -9.946280247742578
glc__D_e_1:  -3.515776907676935
glc__D_e_1:  -0.12019862161922124
glc__D_e_1:  -1.6511679313858187
glc__D_e_1:  -11.415701053226654
glc__D_e_1:  -4.8307532913602165
glc__D_e_1:  -2.2008198493314928
glc__D_e_1:  -7.666129308424613
glc__D_e_1:  -14.187616784265822
glc__D_e_1:  -4.619083985067753
glc__D_e_1:  -2.5671469480353513
glc__D_e_1:  -7.540663888462655
glc__D_e_1:  -9.462111214677165
glc__D_e_1:  -5.706304068341565
glc__D_e_1:  -7.280568387611129
glc__D_e_1:  -10.889891279579288
glc__D_e_1:  -7.8423631559048275
glc__D_e_1:  -4.909674479802532
glc__D_e_1:  -1.6988722077517895
glc__D_e_1:  -2.7380489988525714
glc__D_e_1:  -9.374652824645047
glc__D_e_1:  -6.287519726741957
glc__D_e_1:  -2.2058473581112747
glc__D_e_1:  -2.7532316305462414
glc__D_e_1:  -10.77564919977825
glc__D_e_1:  -9.54009382

glc__D_e_1:  -1.4988493257732456
glc__D_e_1:  -0.009128826751015229
glc__D_e_1:  -0.4827185613823066
glc__D_e_1:  -0.9969481880140232
glc__D_e_1:  -0.33356708391229595
glc__D_e_1:  -1.217688344272711
glc__D_e_1:  -1.0118224469407673
glc__D_e_1:  -0.08382916337828794
glc__D_e_1:  -2.0786827286963536
glc__D_e_1:  -1.101119836662255
strain_number:  0 3.1578947368421053
strain_various:  0 0.9874559494365114
strain_number:  1 1.1578947368421053
strain_various:  1 0.7443229275647869
fd_r_threshold:  [1.05, 49.49285133007527, 7.4326792874974705, 5.89429267447892, 7.359395778529212, 7.128026867202519, 5.435863055662901, 3.741264162410359, 4.632374764976414, 4.331328358136375, 3.497342928748731, 4.755536046765758, 4.08002436978792, 4.375701723559491, 4.875632073153717, 4.391388888888889, 6.434863945578231]
slip_r:  4.831522200193649
18
glc__D_e_1:  -0.44803938047821523
glc__D_e_1:  -1.1820610763991573
glc__D_e_1:  -0.7489280357251689
glc__D_e_1:  -0.9013033551318579
glc__D_e_1:  -0.047878285114

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 49.49285133007527, 7.4326792874974705, 5.89429267447892, 7.359395778529212, 7.128026867202519, 5.435863055662901, 3.741264162410359, 4.632374764976414, 4.331328358136375, 3.497342928748731, 4.755536046765758, 4.08002436978792, 4.375701723559491, 4.875632073153717, 4.391388888888889, 6.434863945578231, 3.9083333333333337, 6.305555555555555, 5.472222222222222, 7.5, 2.0, 4.25, 6.0, 1.0, 2.0, 1.0, 1.0]
slip_r:  2.2
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 49.49285133007527, 7.4326792874974705, 5.89429267447892, 7.359395778529212, 7.128026867202519, 5.435863055662901, 3.741264162410359, 4.632374764976414, 4.331328358136375, 3.497342928748731, 4.755536046765758, 4.08002436978792, 4.375701723559491, 4.875632073153717, 4.391388888888889, 6.43486

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([28,  6, 30, 34, 10, 21,  8,  8, 34, 38, 37, 26, 38, 24, 15,  9, 32,
        0, 15]), 1: array([25, 25,  0, 18, 19, 18, 25, 11, 21, 39, 13, 21, 36, 19, 19, 19, 35,
       17,  6])}
1
strain_number:  0 25.68421052631579
strain_various:  0 14.426046004505174
strain_number:  1 20.473684210526315
strain_various:  1 9.637473031173306
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.42105263157895
strain_various:  0 10.494689331043805
strain_number:  1 20.473684210526315
strain_various:  1 6.369073476802461
fd_r_threshold:  [1.05, 400.33943969135106]
slip_r:  80.9078879382702
3
strain_number:  0 36.10526315789474
strain_various:  0 10.259243518034205
strain_number:  1 20.473684210526315
strain_various:  1 5.1643353987520335
fd_r_threshold:  [1.05, 400.33943969135106, 12.711955590315037]
slip_r:  83.24027905633321
4
strain_number:  0 42.8421052631579
strain_various:  0 11.708131038298653
strain_number:  1 20.526315789473685
strain_various:  1 5.184678044642
fd_r_threshol

glc__D_e_1:  -1.7065417504913216
glc__D_e_1:  -3.237511060257919
glc__D_e_1:  -13.216510100002562
glc__D_e_1:  -10.489162206930395
glc__D_e_1:  -2.98308211014544
glc__D_e_1:  -5.989428975909483
glc__D_e_1:  -12.126014085272038
glc__D_e_1:  -7.392644046902339
glc__D_e_1:  -1.7671845274676243
glc__D_e_1:  -3.789946355900037
glc__D_e_1:  -8.297206785503306
glc__D_e_1:  -6.033192157833521
glc__D_e_1:  -2.168593114800954
glc__D_e_1:  -3.6995624245675516
glc__D_e_1:  -2.2982484016191425
glc__D_e_1:  -1.9893595580175545
glc__D_e_1:  -3.7025403508876327
glc__D_e_1:  -2.8301388210943044
glc__D_e_1:  -9.411531003083645
glc__D_e_1:  -4.987049808315534
glc__D_e_1:  -1.96435111673623
glc__D_e_1:  -3.003527907837012
glc__D_e_1:  -9.942665857347258
glc__D_e_1:  -8.19288085630919
glc__D_e_1:  -5.0330051244822975
glc__D_e_1:  -4.652396113354785
glc__D_e_1:  -9.473366538257471
glc__D_e_1:  -7.054907488786892
glc__D_e_1:  -3.3586222661507863
glc__D_e_1:  -2.9780132550232734
glc__D_e_1:  -17.9097313685596

glc__D_e_1:  -0.4345799995876285
glc__D_e_1:  -0.02187926342576163
glc__D_e_1:  -1.1712788081340515
glc__D_e_1:  -0.1184925865027
glc__D_e_1:  -1.4218330107863242
glc__D_e_1:  -0.44427011875222555
glc__D_e_1:  -0.43778182527927323
glc__D_e_1:  -1.4775466701792426
glc__D_e_1:  -0.49998377814514394
glc__D_e_1:  -2.3022060209887463
glc__D_e_1:  -0.9475075398175088
glc__D_e_1:  -2.130411214881736
strain_number:  0 3.5789473684210527
strain_various:  0 1.425917598330954
strain_number:  1 1.3157894736842106
strain_various:  1 0.7981974151633211
fd_r_threshold:  [1.05, 400.33943969135106, 12.711955590315037, 6.667079171167141, 10.846348299343278, 12.236074118113672, 8.929598726613651, 7.057088878028017, 4.705841003245112, 5.557936918618099, 4.78081895853639, 3.6437235375645978, 4.559739872082834, 4.385138344961855, 3.3952991909562424, 3.916087175610986, 5.65375283446712]
slip_r:  4.382003483615807
18
glc__D_e_1:  -0.5527718373637904
glc__D_e_1:  -0.9370181873261265
glc__D_e_1:  -0.74777727505

glc__D_e_1:  -0.005010122598382294
glc__D_e_1:  -0.1970013227237677
glc__D_e_1:  -0.017557235023623607
glc__D_e_1:  -0.04991153165428497
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 400.33943969135106, 12.711955590315037, 6.667079171167141, 10.846348299343278, 12.236074118113672, 8.929598726613651, 7.057088878028017, 4.705841003245112, 5.557936918618099, 4.78081895853639, 3.6437235375645978, 4.559739872082834, 4.385138344961855, 3.3952991909562424, 3.916087175610986, 5.65375283446712, 7.540555555555555, 3.86, 4.483333333333333, 9.75, 2.111111111111111, 2.25, 3.25, 4.25, 4.0, 5.0]
slip_r:  3.75
28
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 400.33943969135106, 12.711955590315037, 6.667079171167141, 10.846348299343278, 12.236074118113672, 8.929598726613651, 7.057088878028017, 4.705841003245112, 5.557936918618099, 4.7808